# Class 18: Better Than Guessing
### Linear Regression and the Coefficient of Determination

**Record your answers on the paper handout, not here.** Every cell runs as given — no blanks today.

In [ ]:
import numpy as np
from datascience import *
import matplotlib.pyplot as plt
%matplotlib inline

print("Ready!")

### Helper functions

The same four we have been building all along. `slope` and `intercept` come straight from $r$ — that is the whole regression line.

In [ ]:
def standard_units(any_numbers):
    "Convert any array of numbers to standard units."
    return (any_numbers - np.mean(any_numbers)) / np.std(any_numbers)

def correlation(t, label_x, label_y):
    return np.mean(standard_units(t.column(label_x)) * standard_units(t.column(label_y)))

def slope(t, label_x, label_y):
    r = correlation(t, label_x, label_y)
    return r * np.std(t.column(label_y)) / np.std(t.column(label_x))

def intercept(t, label_x, label_y):
    return np.mean(t.column(label_y)) - slope(t, label_x, label_y) * np.mean(t.column(label_x))

---
## Part 1. The Cricket Line

The same fifteen observations from lecture. This cell does the whole calculation you just saw: fit the line, compute $SS_{\text{Total}}$ and $SS_{\text{Residual}}$, and get $R^2$.

In [ ]:
cricket = Table.read_table("./data/cricket_thermometer.csv")
chirps = cricket.column('Chirps_per_sec')
temp   = cricket.column('Temperature_deg_F')

m = slope(cricket, 'Chirps_per_sec', 'Temperature_deg_F')
b = intercept(cricket, 'Chirps_per_sec', 'Temperature_deg_F')
predicted = m * chirps + b
residuals = temp - predicted

SS_total    = np.sum((temp - np.mean(temp)) ** 2)
SS_residual = np.sum(residuals ** 2)
R_squared   = 1 - SS_residual / SS_total
r = correlation(cricket, 'Chirps_per_sec', 'Temperature_deg_F')

print(f"Temperature = {m:.2f} * Chirps_per_sec + {b:.2f}")
print(f"SS_Total = {SS_total:.2f}   SS_Residual = {SS_residual:.2f}")
print(f"R^2 = {R_squared:.4f}     r = {r:.4f}     r^2 = {r**2:.4f}")

plt.figure(figsize=(8, 5))
plt.scatter(chirps, temp, color='steelblue', s=60, zorder=5, label='Observations')
plt.plot(chirps, predicted, color='seagreen', linewidth=2, label='Regression line')
plt.axhline(y=np.mean(temp), color='tomato', linewidth=2, linestyle='--', alpha=0.5,
            label=f'Mean = {np.mean(temp):.1f} F')
plt.xlabel('Chirps per Second'); plt.ylabel('Temperature (F)')
plt.legend(); plt.grid(True, alpha=0.3); plt.show()

> **Handout Q1.1 and Q1.2**

---
## Part 2. Where the Line Stops Working

In [ ]:
for c_ in [19, 40, 0]:
    print(f"{c_:5.1f} chirps/sec  ->  {m * c_ + b:6.1f} F")

print()
print(f"Observed chirp rates ran from {chirps.min():.1f} to {chirps.max():.1f}")

> **Handout Q2.1**

---
## Part 3. A Better Number That Is a Worse Model

A different data set. Seven points, and we fit a straight line to them exactly the way we just did. Look at the plot first.

In [ ]:
x_vals = np.arange(0, 7)
y_vals = x_vals ** 2

data = Table().with_columns('x', x_vals, 'y', y_vals)

m2 = slope(data, 'x', 'y')
b2 = intercept(data, 'x', 'y')
pred2 = m2 * x_vals + b2

plt.figure(figsize=(7, 5))
plt.scatter(x_vals, y_vals, color='steelblue', s=80, zorder=5, label='Data')
plt.plot(x_vals, pred2, color='seagreen', linewidth=2, label=f'y = {m2:.0f}x + ({b2:.0f})')
for xi, yi, yh in zip(x_vals, y_vals, pred2):
    plt.plot([xi, xi], [yi, yh], color='seagreen', alpha=0.5, linewidth=1.5)
plt.xlabel('x'); plt.ylabel('y')
plt.title('Second Data Set')
plt.legend(); plt.grid(True, alpha=0.3); plt.show()

> **Handout Q3.1** — commit to a prediction *before* running the next cell.

In [ ]:
SS_total2    = np.sum((y_vals - np.mean(y_vals)) ** 2)
SS_residual2 = np.sum((y_vals - pred2) ** 2)
R2_second    = 1 - SS_residual2 / SS_total2

print(f"SS_Total = {SS_total2:.0f}   SS_Residual = {SS_residual2:.0f}")
print()
print(f"R^2, second data set = {R2_second:.4f}")
print(f"R^2, crickets        = {R_squared:.4f}")

### The residual plot

Plot the leftover error against x. If the line is the right model, the residuals should scatter with no pattern — the line has already used up everything it can.

In [ ]:
residuals2 = y_vals - pred2

fig, axes = plt.subplots(1, 2, figsize=(13, 4.5))

axes[0].scatter(chirps, residuals, color='steelblue', s=60)
axes[0].axhline(0, color='k', linewidth=1)
axes[0].set_xlabel('Chirps per Second'); axes[0].set_ylabel('Residual (F)')
axes[0].set_title(f'Crickets   (R^2 = {R_squared:.3f})')
axes[0].grid(True, alpha=0.3)

axes[1].scatter(x_vals, residuals2, color='steelblue', s=80)
axes[1].axhline(0, color='k', linewidth=1)
axes[1].set_xlabel('x'); axes[1].set_ylabel('Residual')
axes[1].set_title(f'Second data set   (R^2 = {R2_second:.3f})')
axes[1].grid(True, alpha=0.3)

plt.tight_layout(); plt.show()

> **Handout Q3.2 and Q3.3**